This notebook removes filters samples with a large amount of outliers, and converts the data to a long format of gene-individual pairs.

This notebook should take about 30 seconds to run.

## Setup

In [1]:
library(data.table)

In [2]:
d <- fread('mage_expression-tpm_residuals.csv')

## Filter samples a suspicious amount of high z scores (an amount > 1.5*IQR)
This is meant to remove samples with high expression values which may be due to technical errors.

In [3]:
high_zs_per_sample <- rowSums(d[,-c('sample_id')] > 3)
too_many_high_z_thresh <- quantile(high_zs_per_sample,c(.25,.75)) |> diff()*1.5
d <- d[high_zs_per_sample < too_many_high_z_thresh]

In [4]:
sum(high_zs_per_sample > too_many_high_z_thresh) # Number of samples removed

[1] 197

## Convert to long format of 3 columns: `gene`, `sample_id`, `z`.

In [5]:
d <- melt(d,
  id.vars='sample_id',
  variable.name='gene_id',
  value.name='z'
)

## Filter to genes with at least one outlier (z>3)

In [6]:
genes_w_high_z <- unique(d[z>3,gene_id])
d <- d[gene_id %in% genes_w_high_z]

## Write

In [7]:
setnames(d, c('Ind','gene','MedZ'))
fwrite(d,'mage_expression-watershed_sv_input.tsv', sep='\t')
system('gcloud storage cp mage_expression-watershed_sv_input.tsv $WORKSPACE_BUCKET/data/derived/')